In [ ]:
import os
import requests
import pandas as pd
from pathlib import Path

# --- НАСТРОЙКИ ---
# ID региона (например, 77 - Москва)
REGION_ID = 77 
# Можно добавить фильтр по округу через API или фильтровать уже в коде
# Дата публикации декларации (ГГГГ-ММ-ДД)
START_DATE = '2023-01-01'
END_DATE = '2023-12-31'

# Путь к папке на рабочем столе
desktop = Path.home() / "Desktop"
output_dir = desktop / "Project_Declarations"
output_dir.mkdir(exist_ok=True)

def get_houses():
    url = "https://xn--d1aqf.xn--p1ai"
    params = {
        "objStatus": "0",  # Строящиеся
        "region": REGION_ID,
        "size": 100        # Количество объектов за один запрос
    }
    
    print("Получаем список объектов...")
    response = requests.get(url, params=params)
    if response.status_code == 200:
        return response.json().get('data', {}).get('list', [])
    return []

def download_declaration(obj_id, house_name):
    # API для получения деталей конкретного объекта
    detail_url = f"https://xn--d1aqf.xn--p1ai{obj_id}"
    res = requests.get(detail_url)
    
    if res.status_code == 200:
        data = res.json().get('data', {})
        # Ищем файл проектной декларации
        # Обычно он находится в поле pdId или в списке документов
        pd_id = data.get('pdId')
        
        if pd_id:
            pdf_url = f"https://наш.дом.рф/сервисы/api/serv/files/{pd_id}"
            file_res = requests.get(pdf_url)
            
            if file_res.status_code == 200:
                # Очищаем имя для названия файла
                safe_name = "".join([c for c in str(house_name) if c.isalnum() or c in (' ', '_')]).rstrip()
                file_path = output_dir / f"ID{obj_id}_{safe_name}.pdf"
                
                with open(file_path, 'wb') as f:
                    f.write(file_res.content)
                print(f"Скачано: {safe_name}")

# --- ОСНОВНОЙ ЦИКЛ ---
houses = get_houses()
print(f"Найдено объектов: {len(houses)}")

for house in houses:
    # Пример фильтрации по дате (если доступна в списке)
    pub_date = house.get('objPublDt')
    
    if pub_date and START_DATE <= pub_date <= END_DATE:
        obj_id = house.get('objId')
        addr = house.get('objAddr', f'house_{obj_id}')
        download_declaration(obj_id, addr)

print(f"\nГотово! Файлы сохранены в: {output_dir}")